# Life OS: разбор выгрузки

Шаблон ноутбука. Положи файл ZIP из приложения («Развитие → Аналитика → Выгрузка для анализа») рядом с ноутбуком,
при необходимости поправь имя в первой ячейке и запускай ячейки по порядку.
Нужны пакеты `pandas` и `matplotlib` (`pip install pandas matplotlib`). Данные никуда не отправляются.

In [ ]:
import io
import zipfile
import matplotlib.pyplot as plt
import pandas as pd

ZIP_PATH = "lifeos-analysis.zip"   # имя файла из приложения

with zipfile.ZipFile(ZIP_PATH) as z:
    read = lambda name: pd.read_csv(io.BytesIO(z.read(name)), encoding="utf-8-sig")
    daily = read("daily.csv")
    runs = read("test_runs.csv")

daily["date"] = pd.to_datetime(daily["date"])
daily = daily[daily["sick"].fillna(0) != 1]   # дни режима болезни не берём: они искажают связи
print(f"Дней: {len(daily)}, прохождений проверок: {len(runs)}")
daily.head()

## Результат по дням

`composite` — общий результат проверок в σ относительно твоей же нормы (выше — лучше). Линия — среднее за 7 дней.

In [ ]:
d = daily.dropna(subset=["composite"]).set_index("date")
ax = d["composite"].plot(style=".", alpha=0.5, figsize=(9, 3), label="день")
d["composite"].rolling(7, min_periods=3).mean().plot(ax=ax, label="среднее за 7 дней")
ax.axhline(0, color="gray", lw=0.5)
ax.set_ylabel("σ")
ax.legend()
plt.show()

## Сон и результат

Ранговая корреляция Спирмена: от −1 до +1. Это наблюдение, а не причина.

In [ ]:
pairs = daily[["sleepHours", "composite"]].dropna()
print(f"Дней с обоими значениями: {len(pairs)}")
print(pairs.corr(method="spearman").round(2))
pairs.plot.scatter(x="sleepHours", y="composite", figsize=(5, 4), alpha=0.6)
plt.show()

## Вечерние факторы и результат на следующий день

Кофе, экран, движение вечером дня D могут влиять на проверки дня D+1, поэтому факторы сдвигаем на день.

In [ ]:
factors = ["caffeine", "alcohol", "water", "exercise", "screen", "outdoors", "foodScore", "habitsDone", "focusMin"]
lagged = daily.set_index("date")[factors].shift(1, freq="D")
joined = lagged.join(daily.set_index("date")["composite"], how="inner").dropna(subset=["composite"])
joined.corr(method="spearman")["composite"].drop("composite").round(2).sort_values()

## Дни недели

In [ ]:
d = daily.dropna(subset=["composite"]).copy()
d["weekday"] = d["date"].dt.dayofweek
names = ["пн", "вт", "ср", "чт", "пт", "сб", "вс"]
by = d.groupby("weekday")["composite"].agg(["mean", "count"]).reindex(range(7))
by.index = names
by["mean"].plot.bar(figsize=(6, 3))
plt.show()
by.round(2)

## Осторожно

- Связи в этих таблицах — наблюдения за одним человеком; чем меньше дней, тем легче увидеть закономерность там, где её нет.
- Чтобы проверить причину, измени одну вещь на 1–2 недели и сравни периоды (в приложении есть эксперименты).
- Скрипт `analyze.py` из этой же папки считает то же самое с поправкой на множество сравнений.